# Searching for similar books

Run after `data-exploration.ipynb`. First export the tagged descriptions, then optionally try semantic search. Run from the repository root with the local package installed.


In [ ]:
from pathlib import Path

import pandas as pd

books = pd.read_csv("books_cleaned.csv", dtype={"isbn13": str})
books[["title", "tagged_description"]].head()

## Save one description per line

This export does not call OpenAI. Replace line breaks inside descriptions so each book stays on one line.


In [ ]:
descriptions = []
for _, book in books.iterrows():
    description = " ".join(book["description"].split())
    descriptions.append(f"{book['isbn13']} {description}")

Path("tagged_description.txt").write_text(
    "\n".join(descriptions) + "\n", encoding="utf-8"
)
print(f"Saved {len(descriptions)} descriptions to tagged_description.txt")

## Optional: build the search index

**The following cells send descriptions to OpenAI and make paid API calls.** Set `OPENAI_API_KEY` in the root `.env` first. Restarting this notebook and rebuilding the index embeds the descriptions again. You can stop here if you only need the export for the dashboard.


In [ ]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv()
documents = []
for description in descriptions:
    documents.append(Document(page_content=description))

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
database = Chroma.from_documents(documents, embeddings)

## Try a search

Reuse the app's selection code so the results keep the similarity order and respect the requested number of books.


In [ ]:
from book_recommender.recommendations import select_recommendations

catalogue = {}
for book in books.to_dict("records"):
    catalogue[book["isbn13"]] = book

query = "A book to teach children about nature"
search_results = database.similarity_search(query, k=min(10, len(books)))
recommendations = select_recommendations(catalogue, search_results, limit=10)
pd.DataFrame(recommendations)[["title", "authors", "description"]]